# Module 5: Breathing Light

## Mission Briefing

So far, your LED has been like a light switch — either **fully on** or **fully off**. What if you could make it glow softly, like a candle? Or pulse gently, like a sleeping laptop?

Today, you'll learn a clever trick that lets your Pico create **any brightness level** — even though it can only send 1s and 0s. By the end, your LED will "breathe" in and out smoothly.

```
   Brightness over time:

   bright ╭──╮      ╭──╮      ╭──╮
         ╱    ╲    ╱    ╲    ╱    ╲
   dim  ╱      ╲  ╱      ╲  ╱      ╲
       ╱        ╲╱        ╲╱        ╲
   off ─                              ─
```

Same circuit as Module 2 — but a whole new superpower!

---
## Science Spotlight: The Dimming Trick

Digital means on or off — 1 or 0. But what if you want **HALF** brightness? Your Pico can't send 0.5 volts... or can it fake it?

*Your instructor will reveal the trick that makes dimming possible with only on/off signals.*

---
## Same Circuit as Module 2

Great news — you're using the **exact same wiring** from Module 2! If your breadboard still has the LED circuit, you're ready to go.

### Circuit

```
   Pico GP15 ──── [220 ohm Resistor] ──── LED (+) ──── LED (-) ──── GND
```

### Step-by-Step (if you need to rewire)

1. **Place an LED** on the breadboard (long leg = +, short leg = -)
2. **Connect a 220 ohm resistor** from GP15's row to the LED's long leg row
3. **Connect** the LED's short leg row to **GND**
4. Plug in your USB cable

That's it — same two-component circuit, but today the code makes all the difference.

---
## Code Along: Set Up PWM

Instead of `Pin.OUT` (which only does on/off), we'll use something called **PWM**. This gives us control over brightness.

Fill in the blanks:

In [ ]:
from machine import Pin, PWM

pwm = PWM(Pin(_____))          # Which GPIO pin is the LED on?
pwm.freq(1000)                 # 1000 flickers per second
pwm.duty_u16(_____)            # Try: 0 (off), 32768 (half), 65535 (full)

Run the code with different values in `duty_u16()` and observe the LED:

| Value | What You Should See |
|-------|--------------------|
| `0` | LED completely off |
| `16384` | Dim glow |
| `32768` | About half brightness |
| `49152` | Fairly bright |
| `65535` | Full brightness |

Change the number, run again, and watch the brightness change!

---
## Understanding Duty Cycle

The number you put in `duty_u16()` is the **duty cycle** — it controls what percentage of the time the LED is ON.

```
   duty_u16(65535) = 100% ON         ████████████████

   duty_u16(49152) = 75% ON          ████████████░░░░

   duty_u16(32768) = 50% ON          ████████░░░░░░░░

   duty_u16(16384) = 25% ON          ████░░░░░░░░░░░░

   duty_u16(0)     = 0% ON (off)     ░░░░░░░░░░░░░░░░

   ████ = ON    ░░░░ = OFF
   (this repeats 1000 times per second!)
```

The LED flickers between on and off **so fast** (1000 times per second!) that your eyes can't see the flicker — they just see the average brightness.

**Why 65535?** That's the biggest number that fits in 16 bits (2^16 - 1). The `u16` in `duty_u16` means "unsigned 16-bit."

---
## Code Along: Brightness Levels

Let's step through several brightness levels with a short pause between each. Fill in the blanks:

In [ ]:
from machine import Pin, PWM
import time

pwm = PWM(Pin(15))
pwm.freq(_____)

# Step through brightness levels
pwm.duty_u16(0)               # Off
time.sleep(1)
pwm.duty_u16(_____)           # 25% brightness
time.sleep(1)
pwm.duty_u16(_____)           # 50% brightness
time.sleep(1)
pwm.duty_u16(_____)           # 75% brightness
time.sleep(1)
pwm.duty_u16(65535)           # Full brightness
time.sleep(1)
pwm.duty_u16(0)               # Off again

You should see the LED get brighter in 4 steps, then turn off. Like a slow sunrise!

---
## Code Along: Breathing LED

Now for the main event — a smooth **breathing** effect! The LED will gradually get brighter, then gradually get dimmer, over and over.

We'll use a `for` loop to smoothly change the duty cycle. Fill in the blanks:

In [ ]:
from machine import Pin, PWM
import time

pwm = PWM(Pin(_____))         # LED pin
pwm.freq(1000)

while True:
    # Breathe IN — brightness goes UP
    for brightness in range(0, _____, 256):    # From 0 to max, step 256
        pwm.duty_u16(_______)                  # What variable holds the current brightness?
        time.sleep(0.005)

    # Breathe OUT — brightness goes DOWN
    for brightness in range(65535, _____, -256):  # From max down to 0
        pwm.duty_u16(brightness)
        time.sleep(_____)

Your LED should now glow up and down smoothly — like it's breathing!

**How it works:**
- `range(0, 65535, 256)` counts from 0 to 65535 in steps of 256 (about 256 steps total)
- Each step, we set the brightness and wait 5 milliseconds
- One full "breath" takes about 256 x 0.005 = ~1.3 seconds each way
- Then we count back down for the "exhale"

---
## Experiments

Try these modifications to the breathing code:

1. **Slow breather:** Change `time.sleep(0.005)` to `time.sleep(0.02)`. How does it feel?

2. **Fast breather:** Try `time.sleep(0.001)`. Can you see the individual steps now?

3. **Bigger steps:** Change the step size from `256` to `1024`. What happens to the smoothness?

4. **Half breath:** What if you only go up to `32768` instead of `65535`? Try it.

5. **Nightlight:** Can you make the LED stay at a dim, steady glow (no breathing)? What `duty_u16` value looks good as a nightlight?

---
## Challenge: Custom Breathing Patterns

Choose one (or more!) of these challenges:

### Challenge A: Speed Control
Make the LED breathe at **different speeds**. Start with a slow breath (3 seconds per cycle), then get faster and faster, then slow down again.

**Hint:** The speed depends on the `time.sleep()` value. Can you put the sleep time in a variable and change it?

### Challenge B: Heartbeat
Instead of a smooth in-out, make the LED do a **heartbeat** pattern: two quick pulses, then a pause. Like this:

```
   brightness
     ╱╲  ╱╲
   ──╱  ╲╱  ╲──────────────────
     bump bump    long pause
```

**Hint:** You'll need two short breathing cycles, then a `time.sleep()` for the pause.

### Challenge C: Wave (if you have multiple LEDs)
If you have 2 or more LEDs, make them breathe in a **wave** — when the first LED is bright, the second is dim, and vice versa.

**Hint:** Use two PWM objects on different pins. When one goes up, the other goes down.

Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **PWM (Pulse Width Modulation)** | Flickering a pin on and off very fast to simulate brightness levels |
| **Duty cycle** | The percentage of time the signal is ON during each cycle |
| **Frequency** | How many on/off cycles happen per second (measured in Hz) |
| **Analog** | A signal that can be any value in a range (like dimmer brightness) |
| **Digital** | A signal that is only on or off — 1 or 0, nothing in between |
| **`duty_u16()`** | Sets the duty cycle using a value from 0 (off) to 65535 (fully on) |
| **`range()`** | Creates a sequence of numbers — `range(start, stop, step)` |

---
*Circuit Explorers — Module 5: Breathing Light*